# RTN (W8A16) Quantization via `bitsandbytes` — Full Pipeline

This notebook runs the **complete** RTN quantization study on **Qwen2-1.5B**. RTN (Round-To-Nearest) is the simplest post-training weight quantization strategy: each weight is independently rounded to the nearest representable value in the target integer grid. For this study we use `bitsandbytes` **INT8 loading** (`load_in_8bit=True`) which applies LLM.int8() — a mixed-precision decomposition that keeps outlier features in FP16 while quantizing the rest to INT8. This gives us real INT8 GEMM kernels, making throughput numbers directly comparable across methods.

While RTN lacks the error-compensation sophistication of GPTQ or the activation-aware saliency of AWQ, it serves as a critical **upper-bandwidth, lower-distortion baseline** at 8 bits, allowing us to compare the 4-bit methods against a simpler, higher-precision alternative.

This notebook is **fully self-contained**: every function is defined inline, all dependencies are installed via `pip`, and there are **no imports from `src/`, `_shared.py`, or `config/default.yaml`**. It can be run standalone on Colab, Azure ML, or any GPU environment.

### Configuration

| Parameter | Value |
|-----------|-------|
| **Model** | `Qwen/Qwen2-1.5B` (1.5 B parameters, 28 transformer layers) |
| **Precision** | W8A16 — 8-bit weights, 16-bit activations |
| **Quantization** | `bitsandbytes` LLM.int8() via `BitsAndBytesConfig(load_in_8bit=True)` |
| **Selective denylist** | `llm_int8_skip_modules` for ablation variants |
| **Calibration** | 512 samples from WikiText-2 (train split), each 2048 tokens, seed 42 |
| **Library** | `bitsandbytes` — prebuilt wheels, no compilation |

### Ablation Variants

RTN supports full ablation via the `llm_int8_skip_modules` parameter in `BitsAndBytesConfig`, built programmatically by enumerating all `nn.Linear` modules at runtime:

- **`full_quant`** — quantize every Linear layer (attention + MLP projections) to INT8
- **`attn_only_quant`** — quantize only the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`); MLP layers remain in FP16
- **`mlp_only_quant`** — quantize only the MLP projections (`gate_proj`, `up_proj`, `down_proj`); attention layers remain in FP16

Since RTN at 8-bit introduces less distortion than 4-bit methods, the ablation results here provide a useful reference point for understanding how much of the accuracy degradation in GPTQ is attributable to bit-width vs. module family.

### Evaluation Strategy

All evaluation is done via **HuggingFace `transformers`** natively (`model.generate()` and teacher-forcing), with **no vLLM dependency**. This ensures the notebook runs on any single-GPU setup without needing vLLM’s compilation or tensor-parallel infrastructure.

### Notebook Sections

| # | Section | What It Does |
|---|---------|-------------|
| 1 | **Setup** | Install dependencies, build config dict, define all helper functions inline |
| 2 | **FP16 Baseline + Calibration** | Download and cache the FP16 reference model; build and cache the calibration corpus for reproducibility |
| 3 | **Quantization** | Run bitsandbytes INT8 loading for each of the 3 ablation variants, exporting checkpoints to `artifacts/rtn_w8_perchannel/<variant>/` |
| 4 | **Evaluation** | For each artifact, measure perplexity (teacher-forcing on WikiText-2 test), GSM8K accuracy (8-shot), MATH accuracy (Level-5 competition problems), ARC-Challenge accuracy (0-shot MC), GPQA-Diamond accuracy (0-shot MC), deployment throughput (tok/s, ms/tok, peak VRAM), and checkpoint size (bytes/param) |
| 5 | **Ablation Study** | Compute the composite score `0.25*(GSM8K + MATH + ARC + GPQA)` per variant, then derive `delta_attn` and `delta_mlp` to determine which module family degrades accuracy more under INT8 quantization — visualized as a grouped bar chart |
| 6 | **Layer Diagnostics** | Run a fixed probe batch (32 sequences x 256 tokens) through both FP16 and each RTN artifact with forward hooks, collecting per-layer activation percentiles (p50/p90/p99/p99.9/max), per-layer output MSE, and logit-level metrics (KL divergence, cosine similarity, top-1/top-5 agreement) |
| 7 | **Accuracy vs Bandwidth** | Scatter plot of composite score vs actual bytes/param for all RTN variants — at ~1 byte/param, RTN sits at the high-bandwidth end of the trade-off curve |
| 8 | **Deployment Metrics** | Three-panel scatter: score vs tokens/sec, ms/token, and peak VRAM — quantifies the real inference gains from INT8 compression |
| 9 | **Layer MSE Heatmap** | Heatmap of per-layer MSE (transformer layer index vs module family), revealing which layers accumulate the most quantization error under RTN |
| 10 | **Summary** | Consolidated results table with all metrics per variant |

---
## 1. Setup

Install all required packages (Colab-compatible) and define every function
inline so this notebook is fully self-contained. Uses `bitsandbytes` for
INT8 quantization and `transformers` for all evaluation — **no vLLM, no
compilation required**.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes datasets evaluate \
    pandas pyarrow matplotlib seaborn pyyaml safetensors

In [ ]:
from __future__ import annotations

import json
import logging
import math
import os
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s")
logger = logging.getLogger("rtn_notebook")
sns.set_theme(style="whitegrid", font_scale=1.1)

PROJECT_ROOT = Path.cwd().resolve()
print(f"Working directory: {PROJECT_ROOT}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Inline configuration (replaces config/default.yaml)
# ════════════════════════════════════════════════════════════════

def _resolve_paths(cfg):
    paths = cfg.get("paths", {})
    for key in ("artifacts_dir", "results_dir", "calibration_dir", "plots_dir"):
        if key in paths and not Path(paths[key]).is_absolute():
            paths[key] = str(PROJECT_ROOT / paths[key])

cfg = {
    "base_model": "Qwen/Qwen2-1.5B",
    "calibration": {
        "dataset": "wikitext",
        "dataset_name": "wikitext-2-raw-v1",
        "split": "train",
        "num_samples": 512,
        "max_length": 2048,
        "seed": 42,
    },
    "quant_configs": [
        {"method": "gptq", "bits": 4, "group_size": 128, "desc_act": False},
        {"method": "awq",  "bits": 4, "group_size": 128, "zero_point": True, "version": "GEMM"},
        {"method": "rtn",  "bits": 8, "per_channel": True},
    ],
    "ablation_variants": ["full_quant", "attn_only_quant", "mlp_only_quant"],
    "eval": {
        "perplexity": {
            "dataset": "wikitext", "dataset_name": "wikitext-2-raw-v1",
            "split": "test", "max_length": 512, "stride": 256,
            "max_eval_tokens": 131072,
        },
        "gsm8k":          {"num_fewshot": 8, "num_samples": 300, "max_new_tokens": 256},
        "math":           {"num_samples": 500, "max_new_tokens": 1024},
        "arc_challenge":  {"num_samples": 500, "max_new_tokens": 5},
        "gpqa":           {"max_new_tokens": 5},
    },
    "accuracy_weights": {"gsm8k": 0.25, "math": 0.25, "arc_challenge": 0.25, "gpqa": 0.25},
    "deployment_benchmark": {
        "prompt_lengths": [128, 512, 1024],
        "generation_lengths": [128, 256],
        "batch_size": 1, "warmup_iters": 2, "bench_iters": 5,
    },
    "paths": {
        "artifacts_dir": "artifacts",
        "results_dir": "results",
        "calibration_dir": "calibration_data",
        "plots_dir": "results/plots",
    },
}
_resolve_paths(cfg)

METHOD = "rtn"
VARIANTS = ["full_quant", "attn_only_quant", "mlp_only_quant"]

print(f"Base model : {cfg['base_model']}")
print(f"Method     : {METHOD}")
print(f"Variants   : {VARIANTS}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Module utilities — selective quantization via llm_int8_skip_modules
# ════════════════════════════════════════════════════════════════

ATTN_PATTERN = re.compile(r"self_attn\.(q|k|v|o)_proj")
MLP_PATTERN  = re.compile(r"mlp\.(gate|up|down)_proj")

@dataclass
class ModuleClassification:
    attn:  list[str] = field(default_factory=list)
    mlp:   list[str] = field(default_factory=list)
    other: list[str] = field(default_factory=list)
    @property
    def all_linear(self) -> list[str]:
        return self.attn + self.mlp + self.other

def classify_linear_modules(model: nn.Module) -> ModuleClassification:
    cls = ModuleClassification()
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        if ATTN_PATTERN.search(name):   cls.attn.append(name)
        elif MLP_PATTERN.search(name):  cls.mlp.append(name)
        else:                           cls.other.append(name)
    return cls


def build_skip_modules(classification: ModuleClassification, variant: str) -> list[str]:
    """Build the list of module names to SKIP (keep in FP16) for bitsandbytes.

    BitsAndBytesConfig's `llm_int8_skip_modules` accepts full module names
    that should remain un-quantized.
    """
    if variant == "full_quant":
        return []
    if variant == "attn_only_quant":
        return classification.mlp + classification.other
    if variant == "mlp_only_quant":
        return classification.attn + classification.other
    raise ValueError(f"Unknown ablation variant: {variant}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Calibration data
# ════════════════════════════════════════════════════════════════

def get_calibration_data(cfg, tokenizer=None):
    cal_cfg = cfg["calibration"]
    cache_dir = Path(cfg["paths"]["calibration_dir"])
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"calibration_{cal_cfg['num_samples']}.pt"
    if cache_path.exists():
        return torch.load(cache_path, weights_only=False)
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"], trust_remote_code=True)
    ds = load_dataset(cal_cfg["dataset"], cal_cfg["dataset_name"], split=cal_cfg["split"])
    all_text = "\n\n".join(t for t in ds["text"] if t.strip())
    rng = torch.Generator().manual_seed(cal_cfg["seed"])
    max_len = cal_cfg["max_length"]
    encoded = tokenizer(all_text, return_tensors="pt")
    total_tokens = encoded.input_ids.shape[1]
    samples = []
    starts = torch.randint(0, total_tokens - max_len, (cal_cfg["num_samples"],), generator=rng)
    for start in starts:
        end = start + max_len
        input_ids = encoded.input_ids[:, start:end]
        samples.append({"input_ids": input_ids, "attention_mask": torch.ones_like(input_ids)})
    torch.save(samples, cache_path)
    return samples

In [ ]:
# ════════════════════════════════════════════════════════════════
# Checkpoint size
# ════════════════════════════════════════════════════════════════

def bytes_per_param_from_safetensors(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    st_files = list(checkpoint_dir.glob("*.safetensors"))
    if not st_files:
        bin_files = list(checkpoint_dir.glob("*.bin")) + list(checkpoint_dir.glob("*.pt"))
        if not bin_files:
            logger.warning("No checkpoint files found in %s", checkpoint_dir); return 0.0
        total_bytes = total_params = 0
        for p in bin_files:
            sd = torch.load(str(p), map_location="cpu", weights_only=True)
            for t in sd.values():
                if hasattr(t, "numel"): total_bytes += t.numel() * t.element_size(); total_params += t.numel()
        return total_bytes / max(total_params, 1)
    try:
        from safetensors import safe_open
    except ImportError:
        return 0.0
    total_bytes = total_params = 0
    for p in st_files:
        with safe_open(str(p), framework="pt") as f:
            for key in f.keys():
                t = f.get_tensor(key)
                total_bytes += t.numel() * t.element_size(); total_params += t.numel()
    if total_params == 0: return 0.0
    bpp = total_bytes / total_params
    logger.info("Checkpoint %s: %.2f GB, %d params, %.3f bytes/param", checkpoint_dir.name, total_bytes/1e9, total_params, bpp)
    return bpp

In [ ]:
# ════════════════════════════════════════════════════════════════
# RTN quantization via bitsandbytes INT8
# ════════════════════════════════════════════════════════════════

def quantize_rtn(cfg, variant="full_quant"):
    """Quantize the base model to INT8 via bitsandbytes and save the artifact.

    Uses llm_int8_skip_modules for selective quantization (ablation variants).
    """
    output_dir = Path(cfg["paths"]["artifacts_dir"]) / "rtn_w8_perchannel" / variant
    output_dir.mkdir(parents=True, exist_ok=True)

    if (output_dir / "config.json").exists():
        logger.info("RTN artifact already exists at %s — skipping.", output_dir)
        return output_dir

    classification_model = AutoModelForCausalLM.from_pretrained(
        cfg["base_model"], torch_dtype=torch.float16, trust_remote_code=True, device_map="cpu",
    )
    classification = classify_linear_modules(classification_model)
    skip_list = build_skip_modules(classification, variant)
    del classification_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_skip_modules=skip_list if skip_list else None,
    )

    logger.info("Loading model in 8-bit (%s), skipping %d modules...", variant, len(skip_list))
    model = AutoModelForCausalLM.from_pretrained(
        cfg["base_model"],
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"], trust_remote_code=True)

    logger.info("Saving RTN checkpoint to %s", output_dir)
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output_dir

In [ ]:
# ════════════════════════════════════════════════════════════════
# Evaluation functions  (all HuggingFace-native, no vLLM dependency)
# ════════════════════════════════════════════════════════════════

def _load_eval_model(model_path):
    """Load a model for evaluation — works for both FP16 and RTN checkpoints."""
    return AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
    )


def evaluate_perplexity(model_path, cfg, *, device="cuda", **_kw):
    ppl_cfg = cfg["eval"]["perplexity"]
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = _load_eval_model(model_path)
    model.eval()
    ds = load_dataset(ppl_cfg["dataset"], ppl_cfg["dataset_name"], split=ppl_cfg["split"])
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    encodings = tokenizer(text, return_tensors="pt")
    max_len = ppl_cfg["max_length"]
    stride = ppl_cfg["stride"]
    max_eval_tokens = ppl_cfg.get("max_eval_tokens", 131072)
    seq_len = min(encodings.input_ids.size(1), max_eval_tokens)
    nlls, prev_end = [], 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_len, seq_len)
        input_ids = encodings.input_ids[:, begin:end].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :max(0, prev_end - begin)] = -100
        with torch.no_grad():
            nlls.append(model(input_ids, labels=target_ids).loss)
        prev_end = end
        if end >= seq_len:
            break
    ppl = math.exp(torch.stack(nlls).mean().item())
    del model; torch.cuda.empty_cache()
    logger.info("Perplexity (%.0fk tokens): %.2f", seq_len / 1e3, ppl)
    return ppl


# ────────────────────────────────────────────────────────────────
# Shared generation helpers
# ────────────────────────────────────────────────────────────────

_ANSWER_RE = re.compile(r"####\s*(-?[\d,]+\.?\d*)")

def _extract_numeric_answer(text):
    m = _ANSWER_RE.search(text)
    if m:
        return float(m.group(1).replace(",", ""))
    nums = re.findall(r"-?[\d,]+\.?\d*", text)
    return float(nums[-1].replace(",", "")) if nums else None


_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

def _extract_boxed_answer(text):
    matches = _BOXED_RE.findall(text)
    return matches[-1].strip() if matches else None


def _normalize_math_answer(ans):
    if ans is None:
        return None
    ans = ans.strip()
    ans = ans.replace("\\$", "").replace(",", "").replace(" ", "")
    ans = ans.replace("\\frac", "frac").replace("\\dfrac", "frac")
    ans = ans.replace("\\left", "").replace("\\right", "")
    ans = ans.replace("\\text{", "").replace("\\mathrm{", "")
    ans = ans.rstrip("}")
    try:
        return str(float(ans))
    except ValueError:
        return ans.lower()


# ────────────────────────────────────────────────────────────────
# GSM8K  (8-shot, 300 samples)
# ────────────────────────────────────────────────────────────────

_GSM_PROMPT = "Question: {question}\nAnswer: Let's think step by step.\n"

def evaluate_gsm8k(model_path, cfg, **_kw):
    gsm_cfg = cfg["eval"]["gsm8k"]
    n_shot = gsm_cfg.get("num_fewshot", 8)
    num_samples = gsm_cfg.get("num_samples", 300)
    max_new = gsm_cfg.get("max_new_tokens", 256)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = _load_eval_model(model_path); model.eval()
    ds = load_dataset("openai/gsm8k", "main", split="test")
    ds = ds.shuffle(seed=42).select(range(min(num_samples, len(ds))))

    exemplar_ds = load_dataset("openai/gsm8k", "main", split="train")
    exemplars = exemplar_ds.shuffle(seed=42).select(range(n_shot))
    prefix_text = ""
    for ex in exemplars:
        prefix_text += _GSM_PROMPT.format(question=ex["question"]) + ex["answer"] + "\n\n"

    correct = total = 0
    for row in ds:
        full_prompt = prefix_text + _GSM_PROMPT.format(question=row["question"])
        input_ids = tokenizer(
            full_prompt, return_tensors="pt", truncation=True, max_length=4096,
        ).input_ids.to(model.device)
        with torch.inference_mode():
            out_ids = model.generate(
                input_ids=input_ids,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
        pred = _extract_numeric_answer(gen)
        gold = _extract_numeric_answer(row["answer"])
        if pred is not None and gold is not None and abs(pred - gold) < 1e-3:
            correct += 1
        total += 1
    del model; torch.cuda.empty_cache()
    acc = correct / max(total, 1)
    logger.info("GSM8K accuracy: %.3f (%d/%d)", acc, correct, total)
    return acc

# ────────────────────────────────────────────────────────────────
# MATH  (0-shot, 500 Level-5 problems, extract \boxed{} answer)
# ────────────────────────────────────────────────────────────────

_MATH_PROMPT = (
    "Solve the following math problem. "
    "Show your work and put your final answer in \\boxed{{}}.\n\n"
    "Problem: {problem}\n\nSolution:"
)

def evaluate_math(model_path, cfg, **_kw):
    math_cfg = cfg["eval"]["math"]
    num_samples = math_cfg.get("num_samples", 500)
    max_new = math_cfg.get("max_new_tokens", 1024)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = _load_eval_model(model_path); model.eval()
    ds = load_dataset("lighteval/MATH-Hard", split="test")
    ds = ds.shuffle(seed=42).select(range(min(num_samples, len(ds))))

    correct = total = 0
    for row in ds:
        prompt = _MATH_PROMPT.format(problem=row["problem"])
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)
        with torch.inference_mode():
            out_ids = model.generate(
                input_ids=input_ids,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
        pred = _normalize_math_answer(_extract_boxed_answer(gen))
        gold = _normalize_math_answer(_extract_boxed_answer(row["solution"]))
        if pred is not None and gold is not None and pred == gold:
            correct += 1
        total += 1
    del model; torch.cuda.empty_cache()
    acc = correct / max(total, 1)
    logger.info("MATH accuracy: %.3f (%d/%d)", acc, correct, total)
    return acc


# ────────────────────────────────────────────────────────────────
# ARC-Challenge  (0-shot MC, 500 samples, KV-cache not needed —
#                  short prompts, just batch)
# ────────────────────────────────────────────────────────────────

_ARC_CHOICES = ["A", "B", "C", "D", "E"]

def _format_arc_prompt(row):
    labels = row["choices"]["label"]
    texts  = row["choices"]["text"]
    prompt = row["question"] + "\n"
    for lbl, txt in zip(labels, texts):
        prompt += f"{lbl}. {txt}\n"
    prompt += "Answer:"
    return prompt, labels

def evaluate_arc_challenge(model_path, cfg, **_kw):
    arc_cfg = cfg["eval"]["arc_challenge"]
    num_samples = arc_cfg.get("num_samples", 500)
    max_new = arc_cfg.get("max_new_tokens", 5)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = _load_eval_model(model_path); model.eval()
    ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
    ds = ds.shuffle(seed=42).select(range(min(num_samples, len(ds))))

    correct = total = 0
    for row in ds:
        prompt, labels = _format_arc_prompt(row)
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)
        with torch.inference_mode():
            out_ids = model.generate(
                input_ids=input_ids,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
        pred_letter = gen.split()[0].strip().rstrip(".") if gen.split() else ""
        gold = row["answerKey"]
        if pred_letter.upper() == gold.upper():
            correct += 1
        total += 1
    del model; torch.cuda.empty_cache()
    acc = correct / max(total, 1)
    logger.info("ARC-Challenge accuracy: %.3f (%d/%d)", acc, correct, total)
    return acc


# ────────────────────────────────────────────────────────────────
# GPQA  (0-shot MC, full diamond split ~198 Qs, shuffled choices)
# ────────────────────────────────────────────────────────────────

def _format_gpqa_prompt(row, rng):
    correct = row["Correct Answer"]
    wrong = [row["Incorrect Answer 1"], row["Incorrect Answer 2"], row["Incorrect Answer 3"]]
    options = [correct] + wrong
    rng.shuffle(options)
    correct_idx = options.index(correct)
    letters = ["A", "B", "C", "D"]
    prompt = row["Question"] + "\n"
    for lbl, opt in zip(letters, options):
        prompt += f"{lbl}. {opt}\n"
    prompt += "Answer:"
    return prompt, letters[correct_idx]

def evaluate_gpqa(model_path, cfg, **_kw):
    gpqa_cfg = cfg["eval"]["gpqa"]
    max_new = gpqa_cfg.get("max_new_tokens", 5)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = _load_eval_model(model_path); model.eval()
    ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")

    import random
    rng = random.Random(42)

    correct = total = 0
    for row in ds:
        prompt, gold_letter = _format_gpqa_prompt(row, rng)
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).input_ids.to(model.device)
        with torch.inference_mode():
            out_ids = model.generate(
                input_ids=input_ids,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
        pred_letter = gen.split()[0].strip().rstrip(".") if gen.split() else ""
        if pred_letter.upper() == gold_letter.upper():
            correct += 1
        total += 1
    del model; torch.cuda.empty_cache()
    acc = correct / max(total, 1)
    logger.info("GPQA-Diamond accuracy: %.3f (%d/%d)", acc, correct, total)
    return acc


# ────────────────────────────────────────────────────────────────
# Deployment benchmark  (multi-prompt sweep)
# ────────────────────────────────────────────────────────────────

@dataclass
class BenchmarkResult:
    tokens_per_sec: float
    ms_per_token: float
    peak_vram_gb: float
    prompt_length: int
    generation_length: int


def benchmark_throughput(model_path, cfg, **_kw):
    bc = cfg["deployment_benchmark"]
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = _load_eval_model(model_path); model.eval()

    prompt_lengths = bc.get("prompt_lengths", [128, 512, 1024])
    gen_lengths = bc.get("generation_lengths", [128, 256])
    results = []

    for plen in prompt_lengths:
        for glen in gen_lengths:
            dummy = "Hello " * (plen // 2)
            inputs = tokenizer(dummy, return_tensors="pt", truncation=True, max_length=plen).to(model.device)
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            for _ in range(bc["warmup_iters"]):
                with torch.no_grad():
                    model.generate(**inputs, max_new_tokens=glen, do_sample=False)
            total_tokens = 0
            start = time.perf_counter()
            for _ in range(bc["bench_iters"]):
                with torch.no_grad():
                    out_ids = model.generate(**inputs, max_new_tokens=glen, do_sample=False)
                total_tokens += out_ids.shape[1] - inputs["input_ids"].shape[1]
            elapsed = time.perf_counter() - start
            tok_s = total_tokens / elapsed
            peak = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
            r = BenchmarkResult(
                tokens_per_sec=tok_s,
                ms_per_token=(elapsed / total_tokens) * 1000,
                peak_vram_gb=peak,
                prompt_length=plen,
                generation_length=glen,
            )
            results.append(r)
            logger.info(
                "Bench prompt=%d gen=%d: %.1f tok/s, %.2f ms/tok, %.2f GB",
                plen, glen, tok_s, r.ms_per_token, peak,
            )

    del model; torch.cuda.empty_cache()
    best = max(results, key=lambda r: r.tokens_per_sec)
    return best

In [ ]:
# ════════════════════════════════════════════════════════════════
# Layer diagnostics
# ════════════════════════════════════════════════════════════════

@dataclass
class LayerStats:
    method: str; variant: str; layer_idx: int; module_family: str
    module_name: str; mse: float; p50: float; p90: float; p99: float; p99_9: float; act_max: float

@dataclass
class LogitDiagnostics:
    method: str; variant: str; kl_div: float; cosine_sim: float
    top1_agreement: float; top5_agreement: float

def _module_family(name):
    if ATTN_PATTERN.search(name): return "attn"
    if MLP_PATTERN.search(name): return "mlp"
    return "other"

def _layer_index(name):
    m = re.search(r"layers\.(\d+)\.", name)
    return int(m.group(1)) if m else -1

def _activation_percentiles(tensor):
    flat = tensor.float().abs().flatten()
    if flat.numel() == 0: return {"p50": 0, "p90": 0, "p99": 0, "p99_9": 0, "act_max": 0}
    q = torch.quantile(flat, torch.tensor([0.50, 0.90, 0.99, 0.999], device=flat.device))
    return {"p50": q[0].item(), "p90": q[1].item(), "p99": q[2].item(), "p99_9": q[3].item(), "act_max": flat.max().item()}

class _HookCollector:
    def __init__(self): self.data = {}; self._handles = []
    def register(self, model):
        for name, mod in model.named_modules():
            if isinstance(mod, nn.Linear) and _module_family(name) != "other":
                self._handles.append(mod.register_forward_hook(self._make_hook(name)))
    def _make_hook(self, name):
        def hook(mod, inp, out):
            x = inp[0] if isinstance(inp, tuple) else inp
            self.data[name] = {"input": x.detach().cpu(), "output": out.detach().cpu()}
        return hook
    def remove(self):
        for h in self._handles: h.remove()
        self._handles.clear()

def _load_model(path, **_kw):
    return AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
    )

def _compute_logit_diagnostics(fp16_logits, q_logits, method, variant):
    fp = fp16_logits.float().flatten(0, 1); qp = q_logits.float().flatten(0, 1)
    log_p = F.log_softmax(fp, dim=-1); p = log_p.exp()
    log_q = F.log_softmax(qp, dim=-1)
    kl = F.kl_div(log_q, p, reduction="batchmean", log_target=False).item()
    cos = F.cosine_similarity(fp, qp, dim=-1).mean().item()
    top1_agree = (fp.argmax(-1) == qp.argmax(-1)).float().mean().item()
    t5f, t5q = fp.topk(5, dim=-1).indices, qp.topk(5, dim=-1).indices
    top5 = sum(len(set(a.tolist()) & set(b.tolist())) for a, b in zip(t5f, t5q)) / (t5f.size(0) * 5)
    return LogitDiagnostics(method=method, variant=variant, kl_div=kl, cosine_sim=cos, top1_agreement=top1_agree, top5_agreement=top5)

def run_layer_diagnostics(fp16_path, quant_path, cfg, method, variant, *, probe_seqs=32, probe_len=256, device="cuda", **_kw):
    tokenizer = AutoTokenizer.from_pretrained(fp16_path, trust_remote_code=True)
    cal = get_calibration_data(cfg, tokenizer)
    input_ids = torch.cat([s["input_ids"][:, :probe_len] for s in cal[:probe_seqs]], dim=0).to(device)

    fp16_model = _load_model(fp16_path); fp16_model.eval()
    fp16_hooks = _HookCollector(); fp16_hooks.register(fp16_model)
    with torch.no_grad():
        fp16_out = fp16_model(input_ids)
    fp16_logits = fp16_out.logits.detach().cpu()
    fp16_data = dict(fp16_hooks.data); fp16_hooks.remove()
    del fp16_model; torch.cuda.empty_cache()

    q_model = _load_model(quant_path); q_model.eval()
    q_hooks = _HookCollector(); q_hooks.register(q_model)
    with torch.no_grad():
        q_out = q_model(input_ids)
    q_logits = q_out.logits.detach().cpu()
    q_data = dict(q_hooks.data); q_hooks.remove()
    del q_model; torch.cuda.empty_cache()

    stats = []
    for name in sorted(set(fp16_data) & set(q_data)):
        mse = F.mse_loss(q_data[name]["output"].float(), fp16_data[name]["output"].float()).item()
        act = _activation_percentiles(fp16_data[name]["input"])
        stats.append(LayerStats(method=method, variant=variant, layer_idx=_layer_index(name),
                                module_family=_module_family(name), module_name=name, mse=mse, **act))
    logit_diag = _compute_logit_diagnostics(fp16_logits, q_logits, method, variant)
    return stats, logit_diag

In [ ]:
# ════════════════════════════════════════════════════════════════
# Visualization
# ════════════════════════════════════════════════════════════════

def plot_accuracy_vs_bandwidth(eval_df, output_dir):
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    for m in eval_df["method"].unique():
        sub = eval_df[eval_df["method"] == m]
        ax.scatter(sub["bytes_per_param_actual"], sub["score"], label=m.upper(), s=80, edgecolors="k", linewidths=0.5)
        for _, r in sub.iterrows():
            ax.annotate(r["variant"], (r["bytes_per_param_actual"], r["score"]), fontsize=7, ha="left", va="bottom")
    ax.set_xlabel("Effective bytes / parameter"); ax.set_ylabel("Composite accuracy score")
    ax.set_title("Accuracy vs. Bandwidth (Sweet-Spot Plot)"); ax.legend()
    fig.tight_layout(); fig.savefig(output_dir / "accuracy_vs_bandwidth.png", dpi=200); plt.close(fig)

def plot_accuracy_vs_deployment(eval_df, output_dir):
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    metrics = [("tok_s", "Tokens / sec"), ("ms_per_token", "ms / token"), ("peak_vram_gb", "Peak VRAM (GB)")]
    available = [m for m in metrics if m[0] in eval_df.columns]
    if not available: return
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5))
    if len(available) == 1: axes = [axes]
    for ax, (col, label) in zip(axes, available):
        for m in eval_df["method"].unique():
            sub = eval_df[eval_df["method"] == m]
            ax.scatter(sub[col], sub["score"], label=m.upper(), s=80, edgecolors="k", linewidths=0.5)
        ax.set_xlabel(label); ax.set_ylabel("Composite score"); ax.legend(fontsize=8)
    fig.suptitle("Accuracy vs. Deployment Metrics", fontsize=13)
    fig.tight_layout(); fig.savefig(output_dir / "accuracy_vs_deployment.png", dpi=200); plt.close(fig)

def plot_ablation_sensitivity(eval_df, output_dir, task_col="score"):
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for m in eval_df["method"].unique():
        sub = eval_df[eval_df["method"] == m]
        full = sub.loc[sub["variant"] == "full_quant", task_col]
        attn = sub.loc[sub["variant"] == "attn_only_quant", task_col]
        mlp  = sub.loc[sub["variant"] == "mlp_only_quant", task_col]
        if full.empty: continue
        fv = full.values[0]
        if not attn.empty: rows.append({"method": m.upper(), "family": "Attn projections", "delta": fv - attn.values[0]})
        if not mlp.empty:  rows.append({"method": m.upper(), "family": "MLP projections",  "delta": fv - mlp.values[0]})
    if not rows: logger.warning("Not enough ablation data for sensitivity plot."); return
    bar_df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=bar_df, x="method", y="delta", hue="family", ax=ax)
    ax.set_ylabel("Score drop (full_quant - <family>_only)"); ax.set_xlabel("Quantization method")
    ax.set_title("Module-Family Sensitivity: MLP vs Attention")
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    fig.tight_layout(); fig.savefig(output_dir / "ablation_sensitivity.png", dpi=200); plt.close(fig)

def plot_layer_mse_heatmap(layer_stats_df, output_dir):
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    df = layer_stats_df if isinstance(layer_stats_df, pd.DataFrame) else pd.read_parquet(str(layer_stats_df))
    for m in df["method"].unique():
        sub = df[df["method"] == m]
        pivot = sub.pivot_table(index="layer_idx", columns="module_family", values="mse", aggfunc="mean")
        if pivot.empty: continue
        fig, ax = plt.subplots(figsize=(6, max(8, len(pivot) * 0.3)))
        sns.heatmap(pivot, ax=ax, cmap="YlOrRd", linewidths=0.3, cbar_kws={"label": "MSE"})
        ax.set_title(f"Per-Layer MSE - {m.upper()}"); ax.set_ylabel("Transformer layer index"); ax.set_xlabel("Module family")
        fig.tight_layout(); fig.savefig(output_dir / f"layer_mse_{m}.png", dpi=200); plt.close(fig)

In [ ]:
# ════════════════════════════════════════════════════════════════
# Orchestration helpers
# ════════════════════════════════════════════════════════════════

def ensure_fp16(cfg):
    fp16_dir = Path(cfg["paths"]["artifacts_dir"]) / "fp16"
    fp16_dir.mkdir(parents=True, exist_ok=True)
    if (fp16_dir / "config.json").exists():
        logger.info("FP16 reference already cached at %s", fp16_dir)
        return fp16_dir
    logger.info("Saving FP16 reference to %s ...", fp16_dir)
    tok = AutoTokenizer.from_pretrained(cfg["base_model"], trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(cfg["base_model"], torch_dtype=torch.float16, trust_remote_code=True)
    mdl.save_pretrained(str(fp16_dir)); tok.save_pretrained(str(fp16_dir))
    del mdl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return fp16_dir


def discover_artifacts(cfg, method):
    base = Path(cfg["paths"]["artifacts_dir"])
    found = []
    for md in sorted(base.iterdir()):
        if not md.is_dir() or md.name == "fp16":
            continue
        key = md.name.split("_")[0]
        if key != method:
            continue
        for vd in sorted(md.iterdir()):
            if not vd.is_dir() or not any(vd.iterdir()):
                continue
            found.append({"method": key, "variant": vd.name, "path": str(vd)})
    return found


def evaluate_artifact(art, cfg, *, skip_tasks=None):
    skip = skip_tasks or set()
    weights = cfg["accuracy_weights"]
    row = {
        "method": art["method"],
        "variant": art["variant"],
        "bytes_per_param_actual": bytes_per_param_from_safetensors(art["path"]),
    }
    if "perplexity" not in skip:
        try:
            row["ppl"] = evaluate_perplexity(art["path"], cfg)
        except Exception:
            logger.exception("Perplexity failed"); row["ppl"] = None
    if "gsm8k" not in skip:
        try:
            row["gsm8k"] = evaluate_gsm8k(art["path"], cfg)
        except Exception:
            logger.exception("GSM8K failed"); row["gsm8k"] = None
    if "math" not in skip:
        try:
            row["math"] = evaluate_math(art["path"], cfg)
        except Exception:
            logger.exception("MATH failed"); row["math"] = None
    if "arc_challenge" not in skip:
        try:
            row["arc_challenge"] = evaluate_arc_challenge(art["path"], cfg)
        except Exception:
            logger.exception("ARC-Challenge failed"); row["arc_challenge"] = None
    if "gpqa" not in skip:
        try:
            row["gpqa"] = evaluate_gpqa(art["path"], cfg)
        except Exception:
            logger.exception("GPQA failed"); row["gpqa"] = None
    metric_vals = {k: row.get(k) for k in weights}
    if all(v is not None for v in metric_vals.values()):
        row["score"] = sum(weights[k] * metric_vals[k] for k in weights)
    else:
        row["score"] = None
    if "benchmark" not in skip:
        try:
            b = benchmark_throughput(art["path"], cfg)
            row["tok_s"] = b.tokens_per_sec
            row["ms_per_token"] = b.ms_per_token
            row["peak_vram_gb"] = b.peak_vram_gb
        except Exception:
            logger.exception("Benchmark failed")
    return row


def evaluate_all(artifacts, cfg, *, skip_tasks=None):
    return pd.DataFrame([evaluate_artifact(a, cfg, skip_tasks=skip_tasks) for a in artifacts])


def run_all_diagnostics(artifacts, cfg):
    fp16_path = str(Path(cfg["paths"]["artifacts_dir"]) / "fp16")
    all_stats, all_logits = [], []
    for art in artifacts:
        logger.info("Diagnostics: %s / %s", art["method"], art["variant"])
        try:
            s, l = run_layer_diagnostics(fp16_path, art["path"], cfg, art["method"], art["variant"])
            all_stats.extend(s)
            all_logits.append(l)
        except Exception:
            logger.exception("Diagnostics failed for %s / %s", art["method"], art["variant"])
    stats_df = pd.DataFrame([asdict(s) for s in all_stats]) if all_stats else pd.DataFrame()
    return stats_df, [asdict(d) for d in all_logits]


def save_eval_results(eval_df, cfg, method):
    rd = Path(cfg["paths"]["results_dir"]); rd.mkdir(parents=True, exist_ok=True)
    out = rd / f"{method}_eval.jsonl"
    with open(out, "w") as f:
        for row in eval_df.to_dict(orient="records"):
            f.write(json.dumps(row) + "\n")
    logger.info("Saved %d rows to %s", len(eval_df), out)
    return out


def save_diagnostics(layer_stats_df, logit_diags, cfg, method):
    rd = Path(cfg["paths"]["results_dir"]); rd.mkdir(parents=True, exist_ok=True)
    pq = rd / f"{method}_layer_stats.parquet"
    if not layer_stats_df.empty:
        layer_stats_df.to_parquet(str(pq), index=False)
    jp = rd / f"{method}_logit_diagnostics.json"
    with open(jp, "w") as f:
        json.dump(logit_diags, f, indent=2)
    return pq, jp


def get_plots_dir(cfg, method):
    d = Path(cfg["paths"]["plots_dir"]) / method
    d.mkdir(parents=True, exist_ok=True)
    return d

print("All functions defined. Setup complete.")

---
## 2. FP16 Baseline and Calibration Data

In [ ]:
fp16_dir = ensure_fp16(cfg)
print(f"FP16 checkpoint: {fp16_dir}")

tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"], trust_remote_code=True)
get_calibration_data(cfg, tokenizer)
print("Calibration data ready.")

---
## 3. RTN Quantization

Quantize the base model to INT8 per-channel for each ablation variant:

- **full_quant** — quantize all Linear layers
- **attn_only_quant** — quantize only attention projections (q/k/v/o_proj)
- **mlp_only_quant** — quantize only MLP projections (gate/up/down_proj)

Uses `bitsandbytes` with `llm_int8_skip_modules` for the denylist.

In [ ]:
for variant in VARIANTS:
    print(f"\n{'='*60}")
    print(f"Quantizing: RTN / {variant}")
    print(f"{'='*60}")
    try:
        out = quantize_rtn(cfg, variant=variant)
        print(f"  -> Saved to {out}")
    except Exception as e:
        print(f"  !! FAILED: {e}")

print("\nRTN quantization complete.")

---
## 4. Evaluation

For each RTN artifact, run:
- **Perplexity** (teacher-forcing on WikiText-2 test, 128k tokens)
- **GSM8K** accuracy (8-shot, KV-cache prefix reuse, 300 samples)
- **MATH** accuracy (0-shot, Level-5 competition problems, 500 samples)
- **ARC-Challenge** accuracy (0-shot multiple-choice, 500 samples)
- **GPQA-Diamond** accuracy (0-shot multiple-choice, all 198 samples)
- **Deployment benchmark** (tok/s, ms/tok, peak VRAM across prompt lengths)
- **Checkpoint size** (actual bytes/param)

In [ ]:
artifacts = discover_artifacts(cfg, METHOD)
print(f"Found {len(artifacts)} RTN artifact(s):")
for a in artifacts:
    print(f"  {a['variant']:20s}  {a['path']}")

In [ ]:
eval_df = evaluate_all(artifacts, cfg)
save_eval_results(eval_df, cfg, METHOD)
eval_df

---
## 5. Ablation Study — MLP vs Attention Sensitivity

Compare the composite score across ablation variants to determine which
module family (MLP vs attention) is more sensitive to 8-bit RTN
quantization.

- `delta_attn = score(full_quant) - score(attn_only_quant)`
- `delta_mlp  = score(full_quant) - score(mlp_only_quant)`

A **larger delta** means that family causes more degradation.

In [ ]:
plots_dir = get_plots_dir(cfg, METHOD)

plot_ablation_sensitivity(eval_df, plots_dir)

from IPython.display import Image, display
ablation_png = plots_dir / "ablation_sensitivity.png"
if ablation_png.exists():
    display(Image(filename=str(ablation_png)))
else:
    print("Not enough ablation data to generate the sensitivity plot.")

---
## 6. Layer Diagnostics

Run a fixed probe batch (32 sequences x 256 tokens) through both the
FP16 reference and each RTN artifact, collecting:

- Per-layer activation percentiles (p50, p90, p99, p99.9, max)
- Per-layer output MSE (FP16 vs quantized)
- Logit-level KL divergence, cosine similarity, top-1/top-5 agreement

In [ ]:
layer_stats_df, logit_diags = run_all_diagnostics(artifacts, cfg)
save_diagnostics(layer_stats_df, logit_diags, cfg, METHOD)

print(f"Layer stats shape: {layer_stats_df.shape}")
print(f"Logit diagnostics: {len(logit_diags)} entries")

In [ ]:
if logit_diags:
    display(pd.DataFrame(logit_diags))
else:
    print("No logit diagnostics available.")

---
## 7. Accuracy vs Bandwidth Trade-off

Scatter plot of composite accuracy score vs actual bytes per parameter
for all RTN variants, identifying the efficiency sweet spot.

In [ ]:
plot_accuracy_vs_bandwidth(eval_df, plots_dir)

bw_png = plots_dir / "accuracy_vs_bandwidth.png"
if bw_png.exists():
    display(Image(filename=str(bw_png)))

---
## 8. Deployment Metrics

Scatter panels: composite score vs tokens/sec, ms/token, and peak VRAM.
Shows whether bandwidth savings translate into real deployment gains.

In [ ]:
plot_accuracy_vs_deployment(eval_df, plots_dir)

dep_png = plots_dir / "accuracy_vs_deployment.png"
if dep_png.exists():
    display(Image(filename=str(dep_png)))

---
## 9. Layer MSE Heatmap

Heatmap with transformer layer index on the y-axis and module family
(attn/mlp) on the x-axis, coloured by mean MSE. Identifies which layers
accumulate the most quantization error.

In [ ]:
if not layer_stats_df.empty:
    plot_layer_mse_heatmap(layer_stats_df, plots_dir)
    mse_png = plots_dir / f"layer_mse_{METHOD}.png"
    if mse_png.exists():
        display(Image(filename=str(mse_png)))
else:
    print("No layer stats data available for heatmap.")

---
## 10. Summary

Key results for RTN (W8A16, bitsandbytes INT8) on Qwen2-1.5B.

In [ ]:
summary_cols = [
    "variant", "bytes_per_param_actual", "ppl",
    "gsm8k", "math", "arc_challenge", "gpqa", "score",
    "tok_s", "ms_per_token", "peak_vram_gb",
]
available = [c for c in summary_cols if c in eval_df.columns]
display(eval_df[available].round(3))

In [ ]:
print(f"All results saved to: {Path(cfg['paths']['results_dir'])}")
print(f"All plots  saved to: {plots_dir}")
print("\nRTN notebook complete.")